# Final Kaggle Submission - Soil Organic Content

Notebook ini adalah versi bersih untuk submission akhir kompetisi.

Final public-best yang dipakai saat ini adalah:

- File: `submissions/focused_anchor_ridge40.csv`
- Public RMSE: **12.04499**
- Strategy: **60% public-best anchor + 40% ridge correction**

Notebook ini tidak lagi berisi eksperimen batch. Tujuannya hanya:

1. membaca data dan format `sample_submission.csv`,
2. memvalidasi file final `focused_anchor_ridge40.csv`,
3. menulis ulang root `submission.csv` dengan format Kaggle yang benar.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"
TARGET = "property_organic_content"
FINAL_SOURCE = SUBMISSION_DIR / "focused_anchor_ridge40.csv"
FINAL_OUTPUT = ROOT / "submission.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

## Submission History

Catatan public leaderboard dipertahankan agar kita tidak mengulang kandidat yang sudah terbukti lebih buruk.

In [2]:
leaderboard_log = pd.DataFrame([
    {
        "file": "submission.csv (baseline lama)",
        "public_rmse": 12.33438,
        "notes": "Baseline awal dari main notebook lama.",
    },
    {
        "file": "no_source_id.csv",
        "public_rmse": 12.30924,
        "notes": "Mengurangi ketergantungan langsung pada source_id.",
    },
    {
        "file": "no_source_id_raw_log_blend_0.80_linear_calibrated.csv",
        "public_rmse": 12.27152,
        "notes": "Raw/log blend dengan calibration.",
    },
    {
        "file": "ucup_opt_raw_log_core_with_ucup_linear_cal.csv",
        "public_rmse": 12.06924,
        "notes": "Public-best anchor sebelum correction; memakai ucup target-encoded CatBoost + base models.",
    },
    {
        "file": "candidate_balanced.csv",
        "public_rmse": 12.13440,
        "notes": "Gagal membaik karena terlalu jauh dari anchor dan prediksi high-tail lebih compressed.",
    },
    {
        "file": "focused_anchor_ridge40.csv",
        "public_rmse": 12.04499,
        "notes": "Final current best: 60% anchor + 40% ridge correction.",
    },
]).sort_values("public_rmse")

display(leaderboard_log)

,file,public_rmse,notes
5,focused_anchor_ridge40.csv,12.04499,Final current best: 60% anchor + 40% ridge cor...
3,ucup_opt_raw_log_core_with_ucup_linear_cal.csv,12.06924,Public-best anchor sebelum correction; memakai...
4,candidate_balanced.csv,12.13440,Gagal membaik karena terlalu jauh dari anchor ...
2,no_source_id_raw_log_blend_0.80_linear_calibra...,12.27152,Raw/log blend dengan calibration.
1,no_source_id.csv,12.30924,Mengurangi ketergantungan langsung pada source...
0,submission.csv (baseline lama),12.33438,Baseline awal dari main notebook lama.


## Data dan Format Check

Bagian ini memastikan `sample_submission.csv` dan final candidate memiliki format identik.

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
final_submission = pd.read_csv(FINAL_SOURCE)

assert TARGET in train.columns, f"Target column {TARGET!r} tidak ditemukan di train.csv"
assert TARGET not in test.columns, f"Target column {TARGET!r} seharusnya tidak ada di test.csv"
assert list(sample_submission.columns) == ["sample_id", TARGET]
assert list(final_submission.columns) == list(sample_submission.columns)
assert len(final_submission) == len(sample_submission) == len(test)
assert final_submission["sample_id"].equals(sample_submission["sample_id"])
assert final_submission["sample_id"].equals(test["sample_id"])
assert final_submission[TARGET].notna().all()
assert np.isfinite(final_submission[TARGET]).all()

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Final source: {FINAL_SOURCE}")

Train shape: (11210, 52)
Test shape : (2670, 51)
Final source: C:\Users\nicho\Documents\Projects\compfest\submissions\focused_anchor_ridge40.csv


## Prediction Distribution

Distribusi ini dipakai sebagai sanity check. Final candidate sengaja tetap dekat dengan anchor terbaik, tetapi sedikit menaikkan high-tail prediction.

In [4]:
distribution = final_submission[TARGET].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame("focused_anchor_ridge40")
display(distribution)

summary = pd.DataFrame([{
    "file": FINAL_SOURCE.name,
    "public_rmse": 12.04499,
    "mean": final_submission[TARGET].mean(),
    "std": final_submission[TARGET].std(),
    "min": final_submission[TARGET].min(),
    "p01": final_submission[TARGET].quantile(0.01),
    "p05": final_submission[TARGET].quantile(0.05),
    "p50": final_submission[TARGET].quantile(0.50),
    "p95": final_submission[TARGET].quantile(0.95),
    "p99": final_submission[TARGET].quantile(0.99),
    "max": final_submission[TARGET].max(),
}])
display(summary)

,focused_anchor_ridge40
count,"2,670.00000"
mean,34.57482
std,20.70008
min,6.01630
1%,10.92649
5%,13.56131
50%,28.15349
95%,76.36152
99%,108.57104
max,159.15685


,file,public_rmse,mean,std,min,p01,p05,p50,p95,p99,max
0,focused_anchor_ridge40.csv,12.04499,34.57482,20.70008,6.01630,10.92649,13.56131,28.15349,76.36152,108.57104,159.15685


## Write Final `submission.csv`

Root `submission.csv` adalah file yang siap diupload ke Kaggle.

In [5]:
final_submission.to_csv(FINAL_OUTPUT, index=False)
check = pd.read_csv(FINAL_OUTPUT)

assert list(check.columns) == list(sample_submission.columns)
assert len(check) == len(sample_submission)
assert check["sample_id"].equals(sample_submission["sample_id"])
assert check[TARGET].notna().all()
assert np.isfinite(check[TARGET]).all()

print(f"Final Kaggle file written: {FINAL_OUTPUT}")
print("Validation passed.")

Final Kaggle file written: C:\Users\nicho\Documents\Projects\compfest\submission.csv
Validation passed.


## Modeling Recipe Archive

Final file berasal dari eksperimen sebelumnya dengan komposisi berikut:

- Anchor: `ucup_opt_raw_log_core_with_ucup_linear_cal.csv`, public RMSE `12.06924`.
- Correction: ridge meta-model pada cached OOF/test predictions.
- Final blend: `60% anchor + 40% ridge correction`.
- Public RMSE final: **12.04499**.

Eksperimen/cache lama sudah tidak diperlukan untuk menjalankan notebook submission ini. Jika ingin melakukan improvement lagi, buat notebook eksperimen baru agar `main.ipynb` tetap bersih.